# §3 Setup — Question Set Description (v4)

Characterizes the 134-question v4 stimulus set used in the human study.
Covers entity/operator/question-word distributions, yes-no handling, and variant examples.

**Data:** `experiment/s2_v4/s4.csv` | `experiment/s2_v4/s4_question.json`

In [ ]:
import json
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path('/home/david/Desktop/yuna/HPA')
sys.path.insert(0, str(BASE / 'analysis'))

from utils.vqa import VQAAnswerMapper

mapper = VQAAnswerMapper()

# ── Load v4 metadata ──────────────────────────────────────────────────────────
s4 = pd.read_csv(BASE / 'experiment/s2_v4/s4.csv')
q_C = {e['question_id']: e['question_en']
       for e in json.load(open(BASE / 'experiment/s2_v4/s4_question.json'))}
q_B = {e['question_id']: e['question_en']
       for e in json.load(open(BASE / 'experiment/s2_v4/s4_weaker_object.json'))}
q_A = {e['question_id']: e['question_en']
       for e in json.load(open(BASE / 'experiment/s2_v4/s4_pronominalized.json'))}

print(f'v4 question set: {len(s4)} questions')
print(f'Columns: {list(s4.columns)}')

## Distribution Table

In [ ]:
# ── Summary distribution table ────────────────────────────────────────────────
N = len(s4)

print(f'Total questions: {N}\n')

print('=== Entity type ===')
ent_vc = s4['ent'].value_counts()
for ent, cnt in ent_vc.items():
    print(f'  {ent:10s}: {cnt:3d}  ({100*cnt/N:.0f}%)')

print()
print('=== Operator type ===')
op_vc = s4['op'].value_counts()
for op, cnt in op_vc.items():
    print(f'  {op:10s}: {cnt:3d}  ({100*cnt/N:.0f}%)')

print()
print('=== Operator group ===')
opg_vc = s4['op_grp'].value_counts()
for opg, cnt in opg_vc.items():
    print(f'  {opg:10s}: {cnt:3d}  ({100*cnt/N:.0f}%)')

print()
print('=== Question word (w) ===')
w_vc = s4['w'].value_counts()
for w, cnt in w_vc.items():
    print(f'  {w:12s}: {cnt:3d}  ({100*cnt/N:.0f}%)')

print()
print(f'Accuracy (original, pilot): mean={s4["acc_original"].mean():.3f}  std={s4["acc_original"].std():.3f}')
print(f'Signal score:               mean={s4["signal_score"].mean():.3f}  std={s4["signal_score"].std():.3f}')

## Distribution Plots

In [ ]:
ENT_COLORS = {
    'object': '#e74c3c', 'person': '#3498db', 'animal': '#f39c12',
    'food': '#27ae60',   'place': '#1abc9c',  'vehicle': '#2980b9',
    'other': '#9b59b6',  'text': '#e67e22',   'product': '#95a5a6',
}
OP_COLORS = {
    'attr': '#3498db', 'count': '#e74c3c', 'ident': '#2ecc71',
    'spat': '#e67e22', 'exist': '#9b59b6', 'act': '#1abc9c',
    'know': '#f39c12', 'text': '#95a5a6',  'temp': '#34495e',
    'comp': '#7f8c8d', 'cause': '#bdc3c7', 'other': '#ecf0f1',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Entity
ax = axes[0]
ent_order = s4['ent'].value_counts().index.tolist()
colors = [ENT_COLORS.get(e, '#95a5a6') for e in ent_order]
counts = [s4['ent'].value_counts()[e] for e in ent_order]
bars = ax.bar(ent_order, counts, color=colors, edgecolor='white', linewidth=0.5)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(cnt), ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Entity type')
ax.set_ylabel('Questions')
ax.set_title(f'Entity Distribution (N={N})', fontweight='bold')
ax.set_xticklabels(ent_order, rotation=30, ha='right', fontsize=9)

# Operator
ax = axes[1]
op_order = s4['op'].value_counts().index.tolist()
colors_op = [OP_COLORS.get(o, '#95a5a6') for o in op_order]
counts_op = [s4['op'].value_counts()[o] for o in op_order]
bars2 = ax.bar(op_order, counts_op, color=colors_op, edgecolor='white', linewidth=0.5)
for bar, cnt in zip(bars2, counts_op):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(cnt), ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Operator type')
ax.set_ylabel('Questions')
ax.set_title('Operator Distribution', fontweight='bold')
ax.set_xticklabels(op_order, rotation=30, ha='right', fontsize=9)

# Signal score distribution
ax = axes[2]
ax.hist(s4['signal_score'], bins=20, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(s4['signal_score'].mean(), color='#e74c3c', linewidth=1.5,
           linestyle='--', label=f'mean={s4["signal_score"].mean():.3f}')
ax.set_xlabel('Signal score')
ax.set_ylabel('Questions')
ax.set_title('Signal Score Distribution', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Variant Examples

Each question appears in three variants:
- **C (original)** — full, specific question as-is
- **B (weaker object)** — subject/object replaced with a generic noun
- **A (pronominalized)** — subject/object replaced with a pronoun

Variants B and A progressively remove referring expressions, making linguistic priors the
primary remaining signal.

In [ ]:
# ── Show one example per ent type ────────────────────────────────────────────
shown_ents = set()
examples = []
for _, row in s4.iterrows():
    ent = row['ent']
    if ent in shown_ents:
        continue
    qid = row['question_id']
    c = q_C.get(qid, '')
    b = q_B.get(qid, '')
    a = q_A.get(qid, '')
    if c and b and a:
        gt = mapper.get_answers(qid)
        examples.append({
            'ent': ent, 'op': row['op'],
            'C (original)': c,
            'B (weaker obj)': b,
            'A (pronominalized)': a,
            'GT': ', '.join(gt[:3]) if gt else '?',
        })
        shown_ents.add(ent)

ex_df = pd.DataFrame(examples)
pd.set_option('display.max_colwidth', 80)
print(ex_df.to_string(index=False))

## Entity × Operator Coverage Matrix

In [ ]:
# ── Heatmap: ent × op count ───────────────────────────────────────────────────
pivot = s4.groupby(['ent', 'op']).size().unstack(fill_value=0)
ent_order = s4['ent'].value_counts().index.tolist()
op_order = s4['op'].value_counts().index.tolist()
pivot = pivot.reindex(index=ent_order, columns=op_order, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pivot.values, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.columns, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(pivot.index, fontsize=9)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if v > 0:
            ax.text(j, i, str(v), ha='center', va='center', fontsize=8,
                    color='white' if v > pivot.values.max() * 0.5 else 'black')
ax.set_xlabel('Operator')
ax.set_ylabel('Entity')
ax.set_title(f'Entity × Operator Coverage (N={N})', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()